In [0]:


def read_stream(spark_read_options_dict , files_location):
    df= (spark.readStream
         .format("cloudFiles")
         .options(**spark_read_options_dict)
         .load(files_location)
         )
    return df

def write_stream(spark_write_options_dict,df,schema,table):
    (df.writeStream
     .trigger(availableNow=True)
     .outputMode("Append")
     .options(**spark_write_options_dict)
     .toTable(f"workspace.{schema}.{table}")
     .awaitTermination())

In [0]:
from collections import namedtuple
landing_table= namedtuple("landing_table", ["path", "name"])

In [0]:
root ="/Volumes/workspace/bronze_layer/landing/ops/"

In [0]:
table_path_list = [
landing_table(f"{root}customer","customer")
,landing_table(f"{root}inventory","inventory")
,landing_table(f"{root}inventory_descriptions","inventory_descriptions")
,landing_table(f"{root}lookup_tables/uom","uom")
,landing_table(f"{root}store_master","store_master")
,landing_table("/Volumes/workspace/bronze_layer/landing/transactions/","transactions")
]

In [0]:
spark_read_options_dict = {
    "cloudFiles.format": "csv"
    ,"cloudFiles.inferColumnTypes": "true" 
    ,"header": "true" 
    ,"delimiter": ","
    ,"quote": '"'
    ,"escape": '"'
    ,"multiline": "true"
    }
spark_write_options_dict = {
    "delta.columnMapping.mode": "name"
        ,"mergeSchema": "true"
}

In [0]:

for table in table_path_list:
    try:
        spark_read_options_dict_local = spark_read_options_dict.copy()
        spark_read_options_dict_local["cloudFiles.schemaLocation"] = f"/Volumes/workspace/bronze_layer/raw/_schema/{table.name}"
        spark_write_options_dict_local = spark_write_options_dict.copy()
        spark_write_options_dict_local["checkpointLocation"] = f"/Volumes/workspace/bronze_layer/raw/_checkpoint/{table.name}"
        df = read_stream(spark_read_options_dict_local,table.path)
        write_stream(spark_write_options_dict_local,df,"bronze_layer",table.name)
    except Exception as e:
        # print(e)
        print(f"Error in {table.name}")